In [2]:
import re
import random
import os

import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from collections import Counter

device = torch.device('mps' if torch.mps.is_available() else 'cpu')
print('device:', device)

device: mps


In [3]:
DATA_DIR = './data'
train_path = DATA_DIR + '/ratings_train.txt'
test_path = DATA_DIR + '/ratings_test.txt'
train_df = pd.read_table(train_path)
test_df = pd.read_table(test_path)


In [4]:
train_df.head()

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


In [5]:
len(train_df)

150000

### 결측치 삭제

In [6]:
train_df.isnull().sum()


id          0
document    5
label       0
dtype: int64

In [7]:
train_df.dropna(inplace=True)
test_df.dropna(inplace=True)

In [8]:
len(train_df)

149995

In [9]:
sum(train_df.document == '')

0

In [10]:
train_df.info()

<class 'pandas.DataFrame'>
Index: 149995 entries, 0 to 149999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype
---  ------    --------------   -----
 0   id        149995 non-null  int64
 1   document  149995 non-null  str  
 2   label     149995 non-null  int64
dtypes: int64(2), str(1)
memory usage: 4.6 MB


In [11]:
train_df.label.astype(np.int16)

0         0
1         1
2         0
3         0
4         1
         ..
149995    0
149996    1
149997    0
149998    1
149999    0
Name: label, Length: 149995, dtype: int16

In [12]:
train_texts, valid_texts, train_labels, valid_labels = train_test_split(
    train_df.document.tolist(),
    train_df.label.tolist(),
    test_size=0.2,
    random_state=42,
    stratify=train_df.label
)

In [13]:
print("ord('a'):", ord('a'), "\nord('A'):", ord('A'))

ord('a'): 97 
ord('A'): 65


In [14]:
text = train_texts[0].lower()
print('text: \n', text , '\n======================')
text = re.sub(r"[^가-힣ㄱ-ㅎㅏ-ㅣa-z0-9\s]", " ", text)
print('text: \n', text , '\n======================')
text = re.sub(r"\s+", " ", text)       # \s+: 공백 1개 이상
print('text: \n', text , '\n======================')

text: 
 헐리웃에 길들여 져서 cg는 비견 되지 안는다.. 내용은 너무 함축된건지.. 전개가 헛점이 많다.. 스토리 진행이 분명 빠른데 지루하다.. 왜 지? 주성치의 서유기가 영향이 크다.. 견자단은 안 맞는 옷 입은.. 곰 탈 웃겼다.. 2시간 갔네..후.. 
text: 
 헐리웃에 길들여 져서 cg는 비견 되지 안는다   내용은 너무 함축된건지   전개가 헛점이 많다   스토리 진행이 분명 빠른데 지루하다   왜 지  주성치의 서유기가 영향이 크다   견자단은 안 맞는 옷 입은   곰 탈 웃겼다   2시간 갔네  후   
text: 
 헐리웃에 길들여 져서 cg는 비견 되지 안는다 내용은 너무 함축된건지 전개가 헛점이 많다 스토리 진행이 분명 빠른데 지루하다 왜 지 주성치의 서유기가 영향이 크다 견자단은 안 맞는 옷 입은 곰 탈 웃겼다 2시간 갔네 후  


In [15]:
tokens = text.split()

In [16]:
def simple_tokenize(text : str) -> list:
    text = text.lower()
    text = re.sub(r"[^가-힣ㄱ-ㅎㅏ-ㅣa-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    tokens = text.split()
    return tokens


In [17]:
simple_tokenize(train_df.document.iloc[5])

['막', '걸음마', '뗀', '3세부터', '초등학교', '1학년생인', '8살용영화', 'ㅋㅋㅋ', '별반개도', '아까움']

In [18]:
counter = Counter()

for x in train_texts:
    tokens = simple_tokenize(x)
    counter.update(tokens)

In [19]:
min_freq = 2
max_vocab = 20000
sorted_tokens = []

for token, freq in counter.most_common():
    if freq >= min_freq:
        sorted_tokens.append(token)

print(len(sorted_tokens))
print(len(sorted_tokens[: max_vocab]))

60416
20000


In [20]:
sorted_tokens = [x for x, y in counter.most_common() if y >= min_freq][: max_vocab]
sorted_tokens.__len__()

20000

In [21]:
def build_vocab(texts, min_freq=2, max_vocab=20000):
    counter = Counter()
    for x in texts:
        tokens = simple_tokenize(x)
        counter.update(tokens)
    vocab = {"<PAD>": 0, "<UNK>": 1}
    sorted_tokens = [token for token, freq in counter.most_common() if freq >= min_freq ][:max_vocab - 2]
    for token in sorted_tokens:
        vocab[token] = len(vocab)
    return vocab

vocab = build_vocab(train_texts)


text = train_texts[0]
tokens = simple_tokenize(text)


In [22]:
token_ids = [vocab.get(token, vocab['<UNK>']) for token in tokens]
token_ids[:4]

[17048, 1, 1, 2097]

In [23]:
import tiktoken
target_model  = 'gpt-4.1-2025-04-14'
tokenizer  = tiktoken.encoding_for_model(target_model)

In [24]:
tokenizer.encode("서울 소프트웨어 아카데미에서 간식을 주셨네요")

[118242,
 22183,
 149651,
 175538,
 12652,
 18124,
 15547,
 14561,
 11440,
 53736,
 104508,
 16130,
 109182,
 88730]

In [25]:
def encode_text(text, vocab, max_len=50):
    tokens = simple_tokenize(text)
    token_ids = [vocab.get(token, vocab['<UNK>']) for token in tokens]
    if len(token_ids) > max_len:
        token_ids = token_ids[:max_len]
    else:
        token_ids += [vocab['<PAD>']] * (max_len - len(token_ids))
    return token_ids

In [26]:
from torch.utils.data import Dataset, DataLoader

In [27]:
class NSMCDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=50):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab
        self.max_len = max_len
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        input_ids = encode_text(text, self.vocab, self.max_len)
        return {
            'input_ids': torch.tensor(input_ids, dtype=torch.long),
            'label': torch.tensor(label, dtype=torch.float32)
        }

In [28]:
max_len = 50
batch_size = 64
train_dataset = NSMCDataset(train_texts, train_labels, vocab, max_len=max_len)
valid_dataset = NSMCDataset(valid_texts, valid_labels, vocab, max_len=max_len)

In [29]:
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)

In [30]:
import torch.nn as nn

class SentimentBiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128, num_layers=1, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=0
        )
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,       # [배치, 길이, 특징]
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, 1)
    
    def forward(self, input_ids):
        embedded = self.embedding(input_ids)
        output, (hidden, cell) = self.lstm(embedded)

        forward_hidden = hidden[-2]     # 마지막 층의 정방향 은닉 상태
        backward_hidden = hidden[-1]    # 마지막 층의 역방향 은닉 상태
        final_hidden = torch.cat((forward_hidden, backward_hidden), dim=1)
        final_hidden = self.dropout(final_hidden)
        logits = self.fc(final_hidden).squeeze(1)

        return logits

In [31]:
model = SentimentBiLSTM(
    vocab_size=len(vocab),
    embed_dim=128,
    hidden_dim=128,
    num_layers=1,
    dropout=0.3
).to(device)

In [32]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [33]:
def binary_accuracy_from_logits(logits, labels):
    probs = torch.sigmoid(logits)
    preds = (probs > 0.5).float()
    correct = (preds == labels).float().sum()
    acc = correct / labels.size(0)
    return acc

In [34]:
def train_one_epoch(model, data_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    total_acc = 0.0

    for batch in tqdm(data_loader, desc='Train'):
        input_ids = batch['input_ids'].to(device)
        labels = batch['label'].to(device)

        logits = model(input_ids)
        loss = criterion(logits, labels)
        acc = binary_accuracy_from_logits(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_acc += acc.item()
    
    avg_loss = total_loss / len(data_loader)
    avg_acc = total_acc / len(data_loader)
    return avg_loss, avg_acc

In [35]:
def evaluate(model, data_loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_acc = 0.0

    with torch.no_grad():
        for batch in tqdm(data_loader, desc='Eval'):
            input_ids = batch['input_ids'].to(device)
            labels = batch['label'].to(device)

            logits = model(input_ids)
            loss = criterion(logits, labels)
            acc = binary_accuracy_from_logits(logits, labels)

            total_loss += loss.item()
            total_acc += acc.item()
            
    avg_loss = total_loss / len(data_loader)
    avg_acc = total_acc / len(data_loader)
    return avg_loss, avg_acc

In [36]:
num_epochs = 5
best_valid_acc = 0.0

print(f'device: {device}')

for epoch in range(num_epochs):
    print(f'\n===== Epoch {epoch + 1} / {num_epochs} =====')
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    valid_loss, valid_acc = evaluate(model, valid_loader, criterion, device)
    print(f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}')
    print(f'Valid Loss: {valid_loss:.4f} | Valid Acc: {valid_acc:.4f}')
    if valid_acc > best_valid_acc:
        best_valid_acc = valid_acc
        torch.save(model.state_dict(), "best_sentiment_bilstm.pt")
        print("가장 좋은 모델을 저장했습니다.")


device: mps

===== Epoch 1 / 5 =====


Train:   7%|▋         | 131/1875 [00:02<00:26, 65.22it/s]


KeyboardInterrupt: 

In [39]:
import pickle
with open('./model_file.pkl', 'wb') as f:
    pickle.dump(vocab, f)

In [46]:
total_params = sum(p.numel() for p in model.parameters())
total_params

2824449